In [1]:
import matplotlib
%matplotlib qt

import matplotlib.pyplot as plt
# to run GUI event loop
#%matplotlib tk
%autosave 180
#%load_ext autoreload
#%autoreload 2
import numpy as np
import nidaqmx

from nidaqmx.constants import LineGrouping
from nidaqmx.constants import Edge
from nidaqmx.constants import AcquisitionType 
import nidaqmx.stream_writers

import time

Autosaving every 180 seconds


In [2]:
############################################
############################################
############################################
import time

def make_tone(f, amp, duration):
    fs = 200000     # 200kHz    (Sample Rate)
    T = 1/f
    Ts = 1/fs

    x = np.arange(int(fs*duration))
    y = [ amp*np.sin(2*np.pi*f * (i/fs)) for i in x]
    y_new = np.tile(y,1)
    
    return y_new

############################################
############################################
############################################

#  Initialize speaker task;
audio_Task = nidaqmx.Task()

# 
audio_Task.ao_channels.add_ao_voltage_chan('Dev3/ao0')
audio_Task.timing.cfg_samp_clk_timing(rate = 200000, 
                                     #samps_per_chan=100,  # in continuos mode this is the buffer
                                     sample_mode=nidaqmx.constants.AcquisitionType.CONTINUOUS)
                                     #sample_mode=nidaqmx.constants.AcquisitionType.FINITE)

# 
audio_Writer = nidaqmx.stream_writers.AnalogSingleChannelWriter(audio_Task.out_stream, 
                                                               auto_start=True)

# realtime streaming with callback
amp = 0.1  # tone amplitude in ?
duration = 0.1 # tone duration in seconds;
start = time.time()
ctr=0
freq = 10000
fname_freq =  r"F:\freq.npy"
while True:       
    tone = make_tone(freq, amp, duration)
    audio_Writer.write_many_sample(tone)

    # load frequency from disk
    freq = np.load(fname_freq)

#    
audio_Task.stop()
audio_Task.close()

print ("DONE")

ValueError: Cannot load file containing pickled data when allow_pickle=False

In [3]:
audio_Task.stop()
audio_Task.close()

In [45]:
########################################
########################################
########################################


import numpy as np
import time
import nidaqmx
from nidaqmx.constants import (AcquisitionType)  # https://nidaqmx-python.readthedocs.io/en/latest/constants.html
import numpy as np
import matplotlib.pyplot as plt
from nidaqmx.constants import TerminalConfiguration
import math
#import ctypes
#libc = ctypes.CDLL('libc.so.6')


class BMI():

    def __init__(self):
        pass

    def play_tone(self):

        # Define variables
        self.sampleRate = 1E3     # Sample rate of NI card
        self.sampleRate_2P = 30	  # Sample rate of BScope 
        self.n_frames_to_be_acquired = 3000   # Number of frames from BScope


        print('Running TONE modulation (ctrl-c to stop)')
        # * Create a DAQmx task named 'softwareTimedVoltage'
        #   More details at: "help dabs.ni.daqmx.Task"
        #   C equivalent - DAQmxCreateTask 
        #   http://zone.ni.com/reference/en-XX/help/370471AE-01/daqmxcfunc/daqmxcreatetask/
        with nidaqmx.Task('hardwareContinuousVoltageWithCallBackNoPlot') as task:

            # * Set up analog input 0 on device defined by variable devName
            #   It's is also valid to use device and channel only: task.ai_channels.add_ai_voltage_chan('Dev1/ai0');
            #   But see defaults: help(task.ai_channels.add_ai_voltage_chan)
            #   C equivalent - DAQmxCreateAIVoltageChan
            #   http://zone.ni.com/reference/en-XX/help/370471AE-01/daqmxcfunc/daqmxcreateaivoltagechan/
            #task.ai_channels.add_ai_voltage_chan('Dev3/ai0')
            task.ai_channels.add_ai_voltage_chan("Dev3/ai0", terminal_config = TerminalConfiguration.NRSE)


            # * Configure the sampling rate and the number of samples
            #   C equivalent - DAQmxCfgSampClkTiming
            #   http://zone.ni.com/reference/en-XX/help/370471AE-01/daqmxcfunc/daqmxcfgsampclktiming/
            #   More details at: "help(task.cfg_samp_clk_timing)
            #   https://nidaqmx-python.readthedocs.io/en/latest/constants.html
            task.timing.cfg_samp_clk_timing(self.sampleRate,
                                            #samps_per_chan=pointsToPlot*2, 
                                            sample_mode=AcquisitionType.CONTINUOUS)

            # * Register a callback funtion to be run every N samples
            #task.register_every_n_samples_acquired_into_buffer_event(pointsToPlot,
            #															 pullDataAndPlot)

            # We configured no triggers, so the acquisition starts as soon as hTask.start is run
            # Start the task and plot the data
            task.start()

            #while True:
            t_end = (time.perf_counter() + 
                     self.n_frames_to_be_acquired/self.sampleRate_2P+
                     5) 
            
            
            

(9000, 512, 512)


In [16]:
########################################
########################################
########################################
fig = plt.figure()
ax=plt.subplot(121)
plt.imshow(data[1])
ax=plt.subplot(122)
plt.imshow(data[-1])
plt.show()

TypeError: Invalid shape (1,) for image data

In [116]:
abs_times = np.load(r"D:\User Training\abs_times.npy")
ttl_times = np.load(r"D:\User Training\ttl_times.npy")
ttl_computed = np.load(r"D:\User Training\ttl_computed.npy")
ttl_detected = np.load(r"D:\User Training\ttl_detected.npy")
rois = np.load(r"D:\User Training\rois_traces.npy")
data = np.load(r"D:\User Training\data.npy")
print (len(ttl_computed))
print (len(ttl_detected))
print (data.shape)

#
plt.figure()
ax=plt.subplot(121)
diff = ttl_times[1:]-ttl_times[:-1]#+1E-8
y = np.histogram(diff,bins=np.arange(0.,0.1, 0.0001))
plt.plot(y[1][:-1],y[0])
plt.semilogy()
plt.ylabel("# of frames")
plt.xlabel("time between ttl pulses (sec): default ~33ms")

#
ax=plt.subplot(122)
diff = abs_times[1:]-abs_times[:-1]#+1E-8
y = np.histogram(diff,bins=np.arange(0.,0.1, 0.0001))
idx = np.where(diff<0.000001)[0]
print(idx)

plt.plot(y[1][:-1],y[0])
plt.semilogy()
plt.ylabel("# of frames")
plt.xlabel("time between reads (sec): default 1ms")

plt.show()
#print (temp)

300
300
(105001, 1)
[]


In [121]:
print (rois.shape)
plt.plot(rois.T)
plt.ylim(bottom=0)
plt.show()

(10, 2997)


In [113]:
##################################
import time
print (time.perf_counter())
ts = 9025           #34904  
idx2 = np.arange(ts  -10,ts  +10,1)

print (abs_times[idx2]-abs_times[0])
print (data[idx2])


t=np.arange(data.shape[0])/1E3

plt.figure()
plt.plot(abs_times[idx2]-abs_times[0],
         data[idx2])
plt.show()

22668.0647297
[10.7577904 10.7579116 10.7616459 10.7618937 10.7620209 10.762142
 10.765501  10.7656826 10.7658072 10.7659284 10.769468  10.7696131
 10.7697396 10.7698603 10.7734357 10.7735825 10.7737088 10.77383
 10.7774394 10.7775874]
[[7.31551416e-04]
 [5.09652719e+00]
 [5.07515132e+00]
 [5.05280382e+00]
 [5.02997050e+00]
 [5.10138535e+00]
 [5.07758040e+00]
 [5.05571871e+00]
 [5.03369509e+00]
 [5.09166904e+00]
 [5.08065723e+00]
 [5.05766197e+00]
 [5.03677192e+00]
 [5.03725773e+00]
 [5.08308630e+00]
 [5.06171043e+00]
 [5.03871518e+00]
 [5.01814900e+00]
 [5.08745864e+00]
 [5.06413951e+00]]


In [15]:
print (data)
t=np.arange(data.shape[0])/1E3
plt.plot(t,data)
plt.show()

[[0.00040768]
 [0.00056962]
 [0.00073155]
 ...
 [0.00073155]
 [0.00056962]
 [0.00056962]]


In [65]:
from tqdm import trange

#
print (data.shape)
last_pulse = 0
#prev_min = 0
ctr=0
for k in trange(data.shape[0]):
    temp = data[k].squeeze()
        
    # check of ttl pulse when from high ~5 to low ~0
    if temp<1 and prev_max>=1:
        if k<(last_pulse+30):
            print (k, )
        last_pulse = k
        #.time_since_last_ttl = self.now-self.previous_trigger
        #print ("pluse: ", ctr)
        # make sure we are out of the trough of the BScope TTL pulse
        #  trough should be about 1-2ms long, so 5ms should be more than enough
        #if self.time_since_last_ttl>0.005:
        #    print ('pulse', ctr)
        ctr+=1
            
    prev_max=temp
print ("# detected ttls: ", ctr)

(3599999, 1)


  1%|▊                                                                           | 37991/3599999 [00:00<00:09, 377153.40it/s]

3976
4009
4042
4076
4109
4142
4176
4209
4242
4276
4309
4342
4376
4409
4442
4476
4509
4543
4576
4609
4643
4676
4709
4743
4776
4809
4843
4876
4909
4943
4976
5009
5043
5076
5109
5143
5176
5209
5243
5276
5309
5343
5376
5409
5443
5476
5510
5543
5576
5610
5643
5676
5710
5743
5776
5810
5843
5876
5910
5943
5976
6010
6043
6076
6110
6143
6176
6210
6243
6276
6310
6343
6376
6410
6443
6477
6510
6543
6577
6610
6643
6677
6710
6743
6777
6810
6843
6877
6910
6943
6977
7010
7043
7077
7110
7143
7177
7210
7243
7277
7310
7343
7377
7410
7444
7477
7510
7544
7577
7610
7644
7677
7710
7744
7777
7810
7844
7877
7910
7944
7977
8010
8044
8077
8110
8144
8177
8210
8244
8277
8310
8344
8377
8411
8444
8477
8511
8544
8577
8611
8644
8677
8711
8744
8777
8811
8844
8877
8911
8944
8977
9011
9044
9077
9111
9144
9177
9211
9244
9278
9311
9344
9378
9411
9444
9478
9511
9544
9578
9611
9644
9678
9711
9744
9778
9811
9844
9878
9911
9944
9978
10011
10044
10078
10111
10144
10178
10211
10245
10278
10311
10345
10378
10411
10445
10478
10511

66167
66200
66234
66267
66300
66334
66367
66400
66434
66467
66500
66534
66567
66600
66634
66667
66700
66734
66767
66800
66834
66867
66900
66934
66967
67001
67034
67067
67101
67134
67167
67201
67234
67267
67301
67334
67367
67401
67434
67467
67501
67534
67567
67601
67634
67667
67701
67734
67768
67801
67834
67868
67901
67934
67968
68001
68034
68068
68101
68134
68168
68201
68234
68268
68301
68334
68368
68401
68434
68468
68501
68535
68568
68601
68635
68668
68701
68735
68768
68801
68835
68868
68901
68935
68968
69001
69035
69068
69101
69135
69168
69201
69235
69268
69301
69335
69368
69402
69435
69468
69502
69535
69568
69602
69635
69668
69702
69735
69768
69802
69835

  3%|██▎                                                                        | 112628/3599999 [00:00<00:09, 367935.49it/s]

...

 81%|███████████████████████████████████████████████████████████▉              | 2913603/3599999 [00:08<00:01, 369401.78it/s]


2854140
2854174
2854207
2854240
2854274
2854307
2854340
2854374
2854407
2854440
2854474
2854507
2854540
2854574
2854607
2854641
2854674
2854707
2854741
2854774
2854807
2854841
2854874
2854907
2854941
2854974
2855007
2855041
2855074
2855107
2855141
2855174
2855207
2855241
2855274
2855308
2855341
2855374
2855408
2855441
2855474
2855508
2855541
2855574
2855608
2855641
2855674
2855708
2855741
2855774
2855808
2855841
2855874
2855908
2855941
2855974
2856008
2856041
2856075
2856108
2856141
2856175
2856208
2856241
2856275
2856308
2856341
2856375
2856408
2856441
2856475
2856508
2856541
2856575
2856608
2856641
2856675
2856708
2856742
2856775
2856808
2856842
2856875
2856908
2856942
2856975
2857008
2857042
2857075
2857108
2857142
2857175
2857208
2857242
2857275
2857308
2857342
2857375
2857409
2857442
2857475
2857509
2857542
2857575
2857609
2857642
2857675
2857709
2857742
2857775
2857809
2857842
2857875
2857909
2857942
2857975
2858009
2858042
2858075
2858109
2858142
2858176
2858209
2858242
2858276

 82%|████████████████████████████████████████████████████████████▉             | 2965826/3599999 [00:08<00:01, 362477.80it/s]


2927775
2927808
2927842
2927875
2927909
2927942
2927975
2928009
2928042
2928075
2928109
2928142
2928175
2928209
2928242
2928275
2928309
2928342
2928375
2928409
2928442
2928475
2928509
2928542
2928576
2928609
2928642
2928676
2928709
2928742
2928776
2928809
2928842
2928876
2928909
2928942
2928976
2929009
2929042
2929076
2929109
2929142
2929176
2929209
2929242
2929276
2929309
2929343
2929376
2929409
2929443
2929476
2929509
2929543
2929576
2929609
2929643
2929676
2929709
2929743
2929776
2929809
2929843
2929876
2929909
2929943
2929976
2930010
2930043
2930076
2930110
2930143
2930176
2930210
2930243
2930276
2930310
2930343
2930376
2930410
2930443
2930476
2930510
2930543
2930576
2930610
2930643
2930676
2930710
2930743
2930777
2930810
2930843
2930877
2930910
2930943
2930977
2931010
2931043
2931077
2931110
2931143
2931177
2931210
2931243
2931277
2931310
2931343
2931377
2931410
2931444
2931477
2931510
2931544
2931577
2931610
2931644
2931677
2931710
2931744
2931777
2931810
2931844
2931877
2931910


KeyboardInterrupt: 

In [91]:
#################################
print (n_ttl)

y = np.histogram(inter_ttl_time, bins=np.arange(0.02,0.1,0.0001))
#plt.plot(y[1][:-1],y[0]+0.1, label='histogram')
plt.bar(y[1][:-1],y[0]+0.1, 0.0001*.9, label='histogram')
plt.plot([0.0333,0.0333],[0,100000],'--', label='correct value')
plt.semilogy()
plt.xlabel("inter TTL pulse interval as detected (sec)")
plt.ylabel("# of frames")
plt.legend()
plt.title("100000 imaging frames")
plt.show()

99990


In [89]:
print (abs_times)
diff = abs_times[-1]-abs_times[0]
print (diff)

for k in range(1,abs_times.shape[0],1):
    diff = abs_times[k]-abs_times[k-1]
    if diff>0.04:
        print("frame: ", k, diff,  abs_times[k-1],abs_times[k])


[   7.9939753    8.0259641    8.0583482 ... 3342.8388699 3342.8729275
 3342.9085325]
3334.9145572
frame:  1804 0.06768979999999658 68.1166169 68.1843067
frame:  4063 0.06767389999998841 143.4851103 143.5527842
frame:  22721 0.06596360000003187 765.7549864 765.82095
frame:  45671 0.06665120000002389 1531.1603214 1531.2269726
frame:  54291 0.0677035999999589 1818.6656017 1818.7333053
frame:  66853 0.06600220000018453 2237.6371572 2237.7031594
frame:  76798 0.06768740000006801 2569.3313085 2569.3989959
frame:  90561 0.06767800000034185 3028.3529283 3028.4206063
frame:  97659 0.06867060000013225 3265.1007799 3265.1694505
frame:  98276 0.06781929999988279 3285.7108319 3285.7786512


In [ ]:
##################################
print (n_ttl)

y = np.histogram(abs_times, bins=np.arange(0.02,0.1,0.0001))
plt.plot(y[1][:-1],y[0]+0.1, label='histogram')
plt.plot([0.0333,0.0333],[0,100000],'--', label='correct value')
plt.semilogy()
plt.xlabel("inter TTL pulse interval as detected (sec)")
plt.ylabel("# of frames")
plt.legend()
plt.title("100000 imaging frames")
plt.show()